# Previsão de Ações do Setor de Energia com LSTM
**Carteira:** XOM, CVX, SLB, HAL — NYSE  
**Objetivo:** Prever o log-return do dia seguinte (regressão multivariada) e comparar a estratégia gerada contra buy & hold no período de teste (2023–2024).

---
### Estrutura
1. Imports e configuração
2. Coleta de dados
3. Engenharia de features
4. Pipeline de dados — janelas, normalização, split
5. Arquitetura LSTM
6. Treino com early stopping
7. Avaliação estatística (RMSE, MAE, R², acurácia direcional)
8. Estratégia de trading com threshold
9. Comparação com buy & hold
10. Análise de resultados

---
## 1. Imports e Configuração

In [ ]:
# !pip install torch yfinance pandas-ta scikit-learn seaborn matplotlib scipy statsmodels

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

import yfinance as yf
import pandas_ta as ta

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Reprodutibilidade ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}  |  pandas-ta: {ta.version}')

# ── Helper para colunas do pandas-ta ──────────────────────────────────────────
def get_col(df, prefix):
    cols = [c for c in df.columns if str(c).startswith(prefix)]
    if not cols:
        raise KeyError(f"Prefixo '{prefix}' não encontrado. Colunas: {list(df.columns)}")
    return cols[0]

# ── Estilo visual ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
CORES = {'XOM': '#1D9E75', 'CVX': '#534AB7', 'SLB': '#D85A30', 'HAL': '#BA7517'}

print('Imports OK')

---
## 2. Coleta de Dados

In [ ]:
TICKERS_ATIVOS = ['XOM', 'CVX', 'SLB', 'HAL']
TICKERS_MACRO  = ['CL=F', 'BZ=F', 'NG=F']
PERIODOS       = [('2010-01-01', '2018-12-31'), ('2023-01-01', '2024-12-31')]

def baixar_concat(tickers, periodos):
    partes = [yf.download(tickers, start=s, end=e,
                          auto_adjust=True, progress=False)
              for s, e in periodos]
    return pd.concat(partes).sort_index()

raw_ativos = baixar_concat(TICKERS_ATIVOS, PERIODOS)
raw_macro  = baixar_concat(TICKERS_MACRO,  PERIODOS)

# Guarda o índice de datas do período de teste (para plots depois)
DATAS_TESTE = raw_ativos.index[raw_ativos.index >= '2023-01-01']

print(f'Ativos: {len(raw_ativos)} dias  |  Macro: {len(raw_macro)} dias')
print(f'Período: {raw_ativos.index[0].date()} → {raw_ativos.index[-1].date()}')
print(f'Dias no período de teste: {len(DATAS_TESTE)}')

---
## 3. Engenharia de Features

13 features divididas em 5 grupos: preço, volume, tendência, momentum, volatilidade e macro.  
Todas as features de preço usam log-return para garantir estacionariedade (ADF p < 0.05).  
`get_col()` detecta nomes de colunas do pandas-ta automaticamente, independente da versão.

In [ ]:
FEATURES = [
    'logret_close', 'logret_open',       # preço
    'volume_ratio',                       # volume
    'sma_diff', 'ema_diff',              # tendência
    'rsi14', 'macd_hist',                # momentum
    'atr14', 'bb_pct_b',                 # volatilidade
    'wti_logret', 'brent_logret',        # macro — petróleo
    'spread_wb', 'ng_logret',            # macro — derivados
]

def calcular_features(ticker, raw_ativos, raw_macro):
    df     = pd.DataFrame(index=raw_ativos.index)
    close  = raw_ativos['Close'][ticker]
    open_  = raw_ativos['Open'][ticker]
    high   = raw_ativos['High'][ticker]
    low    = raw_ativos['Low'][ticker]
    volume = raw_ativos['Volume'][ticker]

    # Preço
    df['logret_close'] = np.log(close / close.shift(1))
    df['logret_open']  = np.log(open_ / close.shift(1))

    # Volume
    df['volume_ratio'] = volume / volume.rolling(20).mean()

    # Tendência
    sma50  = ta.sma(close, length=50)
    sma200 = ta.sma(close, length=200)
    ema12  = ta.ema(close, length=12)
    ema26  = ta.ema(close, length=26)
    df['sma_diff'] = (sma50 - sma200) / sma200
    df['ema_diff'] = (ema12 - ema26)  / close

    # Momentum
    df['rsi14'] = ta.rsi(close, length=14)
    macd_df     = ta.macd(close, fast=12, slow=26, signal=9)
    df['macd_hist'] = macd_df[get_col(macd_df, 'MACDh')] / close

    # Volatilidade
    df['atr14'] = ta.atr(high, low, close, length=14) / close
    bb = ta.bbands(close, length=20, std=2)
    df['bb_pct_b'] = bb[get_col(bb, 'BBP')]

    # Macro
    wti = np.log(raw_macro['Close']['CL=F'] / raw_macro['Close']['CL=F'].shift(1))
    brt = np.log(raw_macro['Close']['BZ=F'] / raw_macro['Close']['BZ=F'].shift(1))
    ng  = np.log(raw_macro['Close']['NG=F'] / raw_macro['Close']['NG=F'].shift(1))
    df['wti_logret']   = wti
    df['brent_logret'] = brt
    df['ng_logret']    = ng
    df['spread_wb']    = wti - brt

    # Target: log-return do dia SEGUINTE
    df['target'] = df['logret_close'].shift(-1)

    return df.dropna()


features_ativos = {t: calcular_features(t, raw_ativos, raw_macro)
                   for t in TICKERS_ATIVOS}

for t, df in features_ativos.items():
    print(f'{t}: {len(df)} amostras')

---
## 4. Pipeline de Dados

### Split temporal
```
Treino    (2010–2016)  →  70%
Validação (2017–2018)  →  15%
Teste     (2023–2024)  →  15%
```
Dados **nunca são embaralhados** — o split respeita a ordem temporal para evitar data leakage.

### Normalização
`StandardScaler` é ajustado **apenas no treino** e aplicado em validação e teste. Isso impede que informação do futuro vaze para a normalização.

In [ ]:
LOOKBACK  = 20    # janela de entrada (1 mês de pregão)
BATCH     = 64

def split_temporal(df):
    """Divide em treino/validação/teste pelo índice de datas."""
    treino = df[df.index <  '2017-01-01']
    val    = df[(df.index >= '2017-01-01') & (df.index < '2019-01-01')]
    teste  = df[df.index >= '2023-01-01']
    return treino, val, teste


def criar_janelas(X, y, lookback):
    """
    Transforma séries em janelas deslizantes.
    Entrada: (T, F) → Saída X: (T-lookback, lookback, F), y: (T-lookback,)
    """
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


class SerieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def preparar_dados(features_ativos, tickers, lookback, batch):
    """
    Junta os 4 ativos, normaliza e cria DataLoaders.
    Retorna também os arrays de teste não normalizados (para avaliação).
    """
    splits = {t: split_temporal(features_ativos[t]) for t in tickers}

    # ── Normalização: scaler ajustado APENAS no treino ────────────────────────
    scalers = {}
    dados   = {'treino': {}, 'val': {}, 'teste': {}}

    for t in tickers:
        tr, va, te = splits[t]
        sc = StandardScaler()
        X_tr = sc.fit_transform(tr[FEATURES].values)
        X_va = sc.transform(va[FEATURES].values)
        X_te = sc.transform(te[FEATURES].values)
        scalers[t] = sc

        y_tr = tr['target'].values
        y_va = va['target'].values
        y_te = te['target'].values

        dados['treino'][t] = criar_janelas(X_tr, y_tr, lookback)
        dados['val'][t]    = criar_janelas(X_va, y_va, lookback)
        dados['teste'][t]  = criar_janelas(X_te, y_te, lookback)

    # ── Concatena os 4 ativos ─────────────────────────────────────────────────
    def concat_split(split_name):
        Xs = np.concatenate([dados[split_name][t][0] for t in tickers])
        ys = np.concatenate([dados[split_name][t][1] for t in tickers])
        return Xs, ys

    X_tr, y_tr = concat_split('treino')
    X_va, y_va = concat_split('val')
    X_te, y_te = concat_split('teste')

    loader_treino = DataLoader(SerieDataset(X_tr, y_tr), batch_size=batch, shuffle=False)
    loader_val    = DataLoader(SerieDataset(X_va, y_va), batch_size=batch, shuffle=False)
    loader_teste  = DataLoader(SerieDataset(X_te, y_te), batch_size=batch, shuffle=False)

    # Guarda dados de teste por ativo (para avaliação individual)
    dados_teste_por_ativo = {t: dados['teste'][t] for t in tickers}

    print(f'Treino:    {len(X_tr):>5} janelas')
    print(f'Validação: {len(X_va):>5} janelas')
    print(f'Teste:     {len(X_te):>5} janelas')

    return (loader_treino, loader_val, loader_teste,
            dados_teste_por_ativo, scalers)


(loader_treino, loader_val, loader_teste,
 dados_teste_por_ativo, scalers) = preparar_dados(
    features_ativos, TICKERS_ATIVOS, LOOKBACK, BATCH
)

---
## 5. Arquitetura LSTM

```
Entrada  →  LSTM (2 camadas, hidden=128, dropout=0.2)  →  Linear(128, 1)
(batch, 20, 13)                                           (batch, 1)
```

**Por que sem ativação na saída:**  
Log-returns são valores reais contínuos (podem ser +0.03 ou −0.05). Qualquer ativação restringiria o intervalo artificialmente. `nn.Linear` puro permite que o modelo preveja qualquer valor real.

**Função de perda híbrida:**  
`Loss = RMSE + λ · (1 − Sharpe normalizado)`  
RMSE sozinho minimiza erro de previsão mas pode gerar sinais de trading ruins. O termo Sharpe penaliza estratégias com baixo retorno ajustado ao risco.

In [ ]:
class EnergyLSTM(nn.Module):
    def __init__(self, input_size=13, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)   # sem ativação — regressão

    def forward(self, x):
        out, _ = self.lstm(x)           # (batch, seq, hidden)
        return self.fc(out[:, -1, :])   # último passo → (batch, 1)


def sharpe_loss(y_pred, y_true, lamb=0.3, eps=1e-8):
    """
    Função de perda híbrida:
        Loss = RMSE + λ · (1 − Sharpe normalizado)

    Sharpe é calculado sobre os retornos simulados da estratégia:
    retorno do dia = y_true * sign(y_pred)   (compra se ŷ > 0, vende se ŷ < 0)
    """
    rmse = torch.sqrt(torch.mean((y_pred.squeeze() - y_true) ** 2))

    retornos = y_true * torch.sign(y_pred.squeeze())
    sharpe   = retornos.mean() / (retornos.std() + eps)
    sharpe_n = torch.tanh(sharpe)          # normaliza para [-1, 1]

    return rmse + lamb * (1.0 - sharpe_n)


# Instancia o modelo
modelo = EnergyLSTM(
    input_size=len(FEATURES),
    hidden_size=128,
    num_layers=2,
    dropout=0.2
).to(DEVICE)

print(modelo)
total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f'\nParâmetros treináveis: {total_params:,}')

---
## 6. Treino com Early Stopping

In [ ]:
# ── Hiperparâmetros ────────────────────────────────────────────────────────────
LR            = 1e-3
EPOCHS        = 100
PATIENCE      = 10     # early stopping
LAMBDA_SHARPE = 0.3    # peso do termo Sharpe na loss

otimizador = torch.optim.Adam(modelo.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    otimizador, mode='min', patience=5, factor=0.5
)


def treinar_epoca(modelo, loader, otimizador):
    modelo.train()
    total = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        otimizador.zero_grad()
        pred = modelo(X_batch)
        loss = sharpe_loss(pred, y_batch, lamb=LAMBDA_SHARPE)
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        otimizador.step()
        total += loss.item()
    return total / len(loader)


def avaliar(modelo, loader):
    modelo.eval()
    total = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            pred  = modelo(X_batch)
            total += sharpe_loss(pred, y_batch, lamb=LAMBDA_SHARPE).item()
    return total / len(loader)


# ── Loop de treino ─────────────────────────────────────────────────────────────
historico      = {'treino': [], 'val': []}
melhor_val     = float('inf')
melhor_pesos   = None
contador_pacie = 0

for epoch in range(1, EPOCHS + 1):
    loss_tr = treinar_epoca(modelo, loader_treino, otimizador)
    loss_va = avaliar(modelo, loader_val)
    scheduler.step(loss_va)

    historico['treino'].append(loss_tr)
    historico['val'].append(loss_va)

    if loss_va < melhor_val:
        melhor_val   = loss_va
        melhor_pesos = {k: v.clone() for k, v in modelo.state_dict().items()}
        contador_pacie = 0
    else:
        contador_pacie += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_atual = otimizador.param_groups[0]['lr']
        print(f'Época {epoch:>3}  |  treino: {loss_tr:.5f}  |  val: {loss_va:.5f}  |  lr: {lr_atual:.2e}  |  paciência: {contador_pacie}/{PATIENCE}')

    if contador_pacie >= PATIENCE:
        print(f'\nEarly stopping na época {epoch}. Melhor val loss: {melhor_val:.5f}')
        break

# Restaura os melhores pesos
modelo.load_state_dict(melhor_pesos)
print('\nMelhores pesos restaurados.')

In [ ]:
# Curva de aprendizado
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(historico['treino'], label='Treino',    color='#1D9E75', linewidth=1.5)
ax.plot(historico['val'],    label='Validação', color='#534AB7', linewidth=1.5)
ax.axvline(np.argmin(historico['val']), color='#888',
           linestyle='--', linewidth=0.8, label='Melhor val')
ax.set_xlabel('Época')
ax.set_ylabel('Loss (híbrida)')
ax.set_title('Curva de Aprendizado', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('curva_aprendizado.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Avaliação Estatística

Métricas calculadas no **conjunto de teste** (2023–2024) por ativo e agregadas.

In [ ]:
def prever(modelo, X_np):
    """Retorna previsões como numpy array."""
    modelo.eval()
    with torch.no_grad():
        X_t   = torch.from_numpy(X_np).to(DEVICE)
        preds = modelo(X_t).cpu().numpy().squeeze()
    return preds


def acuracia_direcional(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))


resultados = {}
previsoes  = {}

for ticker in TICKERS_ATIVOS:
    X_te, y_te = dados_teste_por_ativo[ticker]
    y_pred = prever(modelo, X_te)
    previsoes[ticker] = {'y_true': y_te, 'y_pred': y_pred}

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    dacc = acuracia_direcional(y_te, y_pred)

    resultados[ticker] = {'RMSE': rmse, 'MAE': mae, 'R²': r2, 'Dir. Acc.': dacc}
    print(f'{ticker}  RMSE={rmse:.5f}  MAE={mae:.5f}  R²={r2:.4f}  Dir.Acc.={dacc:.3f}')

# Tabela resumo
df_resultados = pd.DataFrame(resultados).T.round(4)
print('\n', df_resultados.to_string())

In [ ]:
# Retorno previsto vs real — scatter por ativo
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Retorno Previsto vs Real — Período de Teste (2023–2024)',
             fontsize=12, fontweight='bold')

for ax, ticker in zip(axes.flat, TICKERS_ATIVOS):
    y_true = previsoes[ticker]['y_true']
    y_pred = previsoes[ticker]['y_pred']
    ax.scatter(y_true, y_pred, alpha=0.3, s=12, color=CORES[ticker])
    lim = max(abs(y_true).max(), abs(y_pred).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='Perfeito')
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Retorno real')
    ax.set_ylabel('Retorno previsto')
    dacc = resultados[ticker]['Dir. Acc.']
    r2   = resultados[ticker]['R²']
    ax.set_title(f'{ticker}  —  Dir.Acc.={dacc:.2%}  R²={r2:.3f}',
                 fontweight='bold', color=CORES[ticker])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('scatter_previsao.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Estratégia de Trading com Threshold

**Regra:** opera apenas quando `|ŷ| > threshold` — sinais fracos (próximos de zero) são ignorados.

O threshold é tratado como **hiperparâmetro**: testamos vários valores na **validação** e escolhemos o que maximiza o Sharpe Ratio. Isso evita otimizar no teste (data snooping).

In [ ]:
def retornos_estrategia(y_true, y_pred, threshold):
    """
    Calcula retornos diários da estratégia com threshold.
    - |ŷ| > threshold e ŷ > 0  → compra (retorno = +y_true)
    - |ŷ| > threshold e ŷ < 0  → vende   (retorno = -y_true)
    - |ŷ| <= threshold          → cash    (retorno = 0)
    """
    sinais = np.where(np.abs(y_pred) > threshold, np.sign(y_pred), 0)
    return sinais * y_true


def sharpe_anualizado(retornos, dias_ano=252):
    if retornos.std() < 1e-10:
        return 0.0
    return (retornos.mean() / retornos.std()) * np.sqrt(dias_ano)


def max_drawdown(retornos_acum):
    pico = np.maximum.accumulate(retornos_acum)
    dd   = (retornos_acum - pico) / (pico + 1e-10)
    return dd.min()


# ── Busca do threshold ótimo na validação ──────────────────────────────────────
# threshold=0.0 é excluído propositalmente: com ele o modelo opera todos os dias
# (|ŷ| > 0 é sempre verdadeiro), anulando o efeito de filtragem.
# O objetivo do threshold é justamente ignorar sinais fracos e operar
# apenas quando o modelo tem convicção mínima.
THRESHOLDS = [0.001, 0.002, 0.003, 0.004, 0.005, 0.007, 0.010, 0.015]

# Previsões na validação
previsoes_val = {}
for ticker in TICKERS_ATIVOS:
    X_va, y_va = dados_teste_por_ativo[ticker]  # reutilizamos a função
    # Refaz para validação
    _, val_split, _ = split_temporal(features_ativos[ticker])
    sc = scalers[ticker]
    X_val_np = sc.transform(val_split[FEATURES].values)
    X_val_w, y_val_w = criar_janelas(X_val_np, val_split['target'].values, LOOKBACK)
    y_pred_val = prever(modelo, X_val_w)
    previsoes_val[ticker] = {'y_true': y_val_w, 'y_pred': y_pred_val}

print('Busca de threshold ótimo — Sharpe médio na validação:')
sharpes_val = {}
for thr in THRESHOLDS:
    sharpes = []
    for ticker in TICKERS_ATIVOS:
        yt = previsoes_val[ticker]['y_true']
        yp = previsoes_val[ticker]['y_pred']
        ret = retornos_estrategia(yt, yp, thr)
        sharpes.append(sharpe_anualizado(ret))
    media = np.mean(sharpes)
    sharpes_val[thr] = media
    print(f'  threshold={thr:.3f}  →  Sharpe médio val = {media:.3f}')

THRESHOLD_OTIMO = max(sharpes_val, key=sharpes_val.get)
print(f'\nThreshold ótimo selecionado: {THRESHOLD_OTIMO:.3f}')

# ── Diagnóstico: % de dias em cash por threshold ──────────────────────────────
print('\nDiagnóstico de cobertura (% dias operando vs em cash):')
for thr in THRESHOLDS:
    coberturas = []
    for ticker in TICKERS_ATIVOS:
        yp = previsoes_val[ticker]['y_pred']
        cobertura = np.mean(np.abs(yp) > thr)
        coberturas.append(cobertura)
    media_cob = np.mean(coberturas)
    marca = ' ← selecionado' if thr == THRESHOLD_OTIMO else ''
    print(f'  threshold={thr:.3f}  →  operando {media_cob:.1%} dos dias  |  cash {1-media_cob:.1%} dos dias{marca}')

---
## 9. Comparação com Buy & Hold

Comparamos três estratégias no período de teste (2023–2024):
1. **Modelo LSTM** — opera com o threshold ótimo
2. **Buy & hold individual** — segura cada ativo do início ao fim do teste
3. **Buy & hold carteira** — equal-weight dos 4 ativos

In [ ]:
def calcular_metricas(retornos, nome):
    ret_acum  = np.exp(np.cumsum(retornos)) - 1
    ret_total = ret_acum[-1]
    sharpe    = sharpe_anualizado(retornos)
    mdd       = max_drawdown(np.exp(np.cumsum(retornos)))
    dias_op   = np.sum(retornos != 0)
    dacc_op   = np.nan  # só relevante para o modelo
    return {
        'Estratégia':    nome,
        'Retorno total': f'{ret_total:.2%}',
        'Sharpe':        round(sharpe, 3),
        'Max Drawdown':  f'{mdd:.2%}',
        'Dias operando': int(dias_op),
    }, np.exp(np.cumsum(retornos))


resultados_final = []
curvas           = {}   # curvas de capital acumulado

for ticker in TICKERS_ATIVOS:
    y_true = previsoes[ticker]['y_true']
    y_pred = previsoes[ticker]['y_pred']

    # ── Modelo com threshold ──────────────────────────────────────────────────
    ret_modelo = retornos_estrategia(y_true, y_pred, THRESHOLD_OTIMO)
    m, curva   = calcular_metricas(ret_modelo, f'LSTM ({ticker})')
    m['Ativo'] = ticker
    m['Tipo']  = 'LSTM'
    dias_op    = np.sum(np.abs(y_pred) > THRESHOLD_OTIMO)
    m['% em cash'] = f"{1 - dias_op/len(y_pred):.1%}"
    m['Dir. Acc.'] = f"{acuracia_direcional(y_true[np.abs(y_pred) > THRESHOLD_OTIMO], y_pred[np.abs(y_pred) > THRESHOLD_OTIMO]):.2%}"
    resultados_final.append(m)
    curvas[f'LSTM_{ticker}'] = curva

    # ── Buy & hold individual ─────────────────────────────────────────────────
    ret_bh = y_true   # segura o ativo — retorno = retorno real diário
    m_bh, curva_bh = calcular_metricas(ret_bh, f'Buy & Hold ({ticker})')
    m_bh['Ativo'] = ticker
    m_bh['Tipo']  = 'B&H'
    m_bh['% em cash'] = '0%'
    m_bh['Dir. Acc.'] = '—'
    resultados_final.append(m_bh)
    curvas[f'BH_{ticker}'] = curva_bh

# ── Buy & hold carteira igual ponderada ───────────────────────────────────────
ret_carteira = np.mean(
    [previsoes[t]['y_true'] for t in TICKERS_ATIVOS], axis=0
)
m_cart, curva_cart = calcular_metricas(ret_carteira, 'Buy & Hold Carteira')
m_cart['Ativo']    = 'Carteira'
m_cart['Tipo']     = 'B&H Carteira'
m_cart['% em cash'] = '0%'
m_cart['Dir. Acc.'] = '—'
resultados_final.append(m_cart)
curvas['BH_Carteira'] = curva_cart

# ── LSTM carteira igual ponderada ─────────────────────────────────────────────
ret_lstm_cart = np.mean(
    [retornos_estrategia(previsoes[t]['y_true'],
                         previsoes[t]['y_pred'],
                         THRESHOLD_OTIMO)
     for t in TICKERS_ATIVOS], axis=0
)
m_lc, curva_lc = calcular_metricas(ret_lstm_cart, 'LSTM Carteira')
m_lc['Ativo']    = 'Carteira'
m_lc['Tipo']     = 'LSTM Carteira'
m_lc['% em cash'] = '—'
m_lc['Dir. Acc.'] = '—'
resultados_final.append(m_lc)
curvas['LSTM_Carteira'] = curva_lc

df_final = pd.DataFrame(resultados_final)
print('\n=== RESULTADOS FINAIS — Teste 2023–2024 ===')
print(df_final[['Ativo', 'Tipo', 'Retorno total', 'Sharpe',
                'Max Drawdown', 'Dias operando', 'Dir. Acc.']].to_string(index=False))

---
## 10. Análise de Resultados — Visualizações

In [ ]:
# ── Plot 1: Curvas de capital — LSTM vs B&H por ativo ─────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Curvas de Capital — LSTM (threshold={THRESHOLD_OTIMO}) vs Buy & Hold\n'
             'Período de Teste: 2023–2024', fontsize=12, fontweight='bold')

for ax, ticker in zip(axes.flat, TICKERS_ATIVOS):
    n = len(curvas[f'LSTM_{ticker}'])
    x = np.arange(n)

    ax.plot(x, curvas[f'LSTM_{ticker}'],   color=CORES[ticker],
            linewidth=1.5, label=f'LSTM')
    ax.plot(x, curvas[f'BH_{ticker}'],     color='#999',
            linewidth=1.5, linestyle='--', label='Buy & Hold')
    ax.axhline(1.0, color='#ccc', linewidth=0.7)

    ret_l = curvas[f'LSTM_{ticker}'][-1] - 1
    ret_b = curvas[f'BH_{ticker}'][-1]   - 1
    ax.set_title(f'{ticker}  |  LSTM: {ret_l:+.1%}  vs  B&H: {ret_b:+.1%}',
                 fontweight='bold', color=CORES[ticker])
    ax.set_xlabel('Dias de negociação')
    ax.set_ylabel('Capital normalizado (início = 1)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('curvas_capital_ativos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Carteira agregada — LSTM vs B&H carteira ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Carteira Igual Ponderada — LSTM vs Buy & Hold\nPeríodo de Teste: 2023–2024',
             fontsize=12, fontweight='bold')

# Curvas de capital
n = len(curvas['LSTM_Carteira'])
x = np.arange(n)
axes[0].plot(x, curvas['LSTM_Carteira'], color='#1D9E75', linewidth=2,   label='LSTM Carteira')
axes[0].plot(x, curvas['BH_Carteira'],   color='#999',   linewidth=2,
             linestyle='--', label='Buy & Hold Carteira')
axes[0].axhline(1.0, color='#ccc', linewidth=0.7)
axes[0].set_title('Curva de Capital')
axes[0].set_xlabel('Dias de negociação')
axes[0].set_ylabel('Capital normalizado')
axes[0].legend()

# Drawdown
def drawdown_serie(curva):
    pico = np.maximum.accumulate(curva)
    return (curva - pico) / pico

axes[1].fill_between(x, drawdown_serie(curvas['LSTM_Carteira']),
                     color='#1D9E75', alpha=0.4, label='LSTM')
axes[1].fill_between(x, drawdown_serie(curvas['BH_Carteira']),
                     color='#999',   alpha=0.3, label='Buy & Hold')
axes[1].set_title('Drawdown')
axes[1].set_xlabel('Dias de negociação')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('carteira_lstm_vs_bh.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Comparação de Sharpe e Retorno total ───────────────────────────────
df_plot = df_final[df_final['Ativo'].isin(TICKERS_ATIVOS)].copy()
df_plot['Retorno_num'] = df_plot['Retorno total'].str.rstrip('%').astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparação: LSTM vs Buy & Hold por Ativo', fontsize=12, fontweight='bold')

# Sharpe
for i, ticker in enumerate(TICKERS_ATIVOS):
    sub = df_plot[df_plot['Ativo'] == ticker]
    x_pos = [i - 0.18, i + 0.18]
    sharpes = [sub[sub['Tipo']=='LSTM']['Sharpe'].values[0],
               sub[sub['Tipo']=='B&H']['Sharpe'].values[0]]
    axes[0].bar(x_pos[0], sharpes[0], width=0.32, color=CORES[ticker],  label='LSTM' if i==0 else '')
    axes[0].bar(x_pos[1], sharpes[1], width=0.32, color=CORES[ticker],  label='B&H'  if i==0 else '',
                alpha=0.4)

axes[0].axhline(0, color='#888', linewidth=0.7)
axes[0].axhline(1, color='#1D9E75', linewidth=0.7, linestyle=':', alpha=0.6)
axes[0].set_xticks(range(len(TICKERS_ATIVOS)))
axes[0].set_xticklabels(TICKERS_ATIVOS)
axes[0].set_title('Sharpe Ratio Anualizado')
axes[0].legend(['LSTM (sólido)', 'B&H (transparente)'], fontsize=9)

# Retorno total
for i, ticker in enumerate(TICKERS_ATIVOS):
    sub = df_plot[df_plot['Ativo'] == ticker]
    x_pos = [i - 0.18, i + 0.18]
    rets = [sub[sub['Tipo']=='LSTM']['Retorno_num'].values[0],
            sub[sub['Tipo']=='B&H']['Retorno_num'].values[0]]
    axes[1].bar(x_pos[0], rets[0], width=0.32, color=CORES[ticker])
    axes[1].bar(x_pos[1], rets[1], width=0.32, color=CORES[ticker], alpha=0.4)

axes[1].axhline(0, color='#888', linewidth=0.7)
axes[1].set_xticks(range(len(TICKERS_ATIVOS)))
axes[1].set_xticklabels(TICKERS_ATIVOS)
axes[1].set_title('Retorno Total (%)')

plt.tight_layout()
plt.savefig('comparacao_sharpe_retorno.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Tabela final formatada ─────────────────────────────────────────────────────
print('\n' + '='*75)
print('RESUMO FINAL — Teste 2023–2024')
print('='*75)

cols = ['Ativo', 'Tipo', 'Retorno total', 'Sharpe', 'Max Drawdown', 'Dias operando', '% em cash', 'Dir. Acc.']
print(df_final[cols].to_string(index=False))

# Carteira
print('\n--- Carteira igual ponderada ---')
cart = df_final[df_final['Ativo'] == 'Carteira']
print(cart[cols].to_string(index=False))

print(f'\nThreshold utilizado: {THRESHOLD_OTIMO} (selecionado por Sharpe na validação)')
print(f'Nota: Sharpe anualizado assume 252 dias de pregão por ano.')